In [2]:
import math
import numpy as np
import pandas as pd

DB_PATH = "runs_index.csv"
DB = pd.read_csv(DB_PATH, index_col=0)

PROBLEMS = ['vanderpol', 'pollu', 'rober', 'orego', 'hires', 'davis-skodje']

# Same base filtering as final_mainablation_results.ipynb: keep only rows with
# a completed evaluation, and restrict to the ablation-1 seed range (20-29).
# That range only ever used derivmatch pretraining (checked below), so the
# explicit filter is just documentation/safety.
DB = DB[DB['E_VF_test'].notnull()]
DB = DB[(DB['seed'] >= 20) & (DB['seed'] <= 29)]
DB = DB[DB['pretraining'] == 'derivmatch']

# When a problem/model has two different parameter counts on record (an
# earlier capacity-matching pass), keep only the larger (matched-capacity)
# one, exactly as final_mainablation_results.ipynb does.
max_n_params = DB.groupby(['problem', 'model'])['n_params'].transform('max')
DB = DB[DB['n_params'] == max_n_params]

SEEDS = list(range(20, 30))
N_ATTEMPTED = len(SEEDS)  # every cell below had exactly this many seeds submitted

DB.shape

(311, 46)

### Defining "blown up"

`E_trajectory_test` is a normalized squared relative error,
`sum((Yhat - Ytrue)**2) / sum(Ytrue**2)` averaged over species
(`src/utils/eval.jl::relative_error`). That normalization gives it a
built-in, parameter-free reference point: **predicting all zeros scores
exactly 1.0** (`Yhat = 0` makes `per_species = sum(Ytrue**2)/sum(Ytrue**2)
= 1` identically). So `E_trajectory_test < 1` means "better than the
trivial zero predictor," and `>= 1` means the rollout has diverged from
anything resembling the true trajectory.

We also tried defining "blown up" relative to each (problem, model,
training) cell's own minimum (e.g. "more than 100x the best seed in this
cell"), since that's the more obvious first idea. It breaks down exactly
where it matters most, though: `GELU-scaled + shooting` on `rober` never
once produced a usable fit in this data (its *best* seed still scores
`E_trajectory_test = 2.6e18`) — but a factor-of-100-over-the-minimum rule
would still call several of those catastrophic runs "successful," purely
because they happen to be close to each other, all equally broken. The
same problem shows up for `hires` (best seed still `3.7e3`) and several
other cells. A threshold relative to the group's own minimum has no way
to tell "a tight cluster of good fits" from "a tight cluster of garbage."

Anchoring to the metric's own zero-predictor baseline avoids that
entirely — it doesn't care what the rest of the group did, so it can't be
fooled by a uniformly-bad group. So:

**A run counts as successful iff `0 <= E_trajectory_test < 1`.**

(No row in this data has a NaN `E_trajectory_test` — the ODE rollout never
hit `successful_retcode == false` for a run that made it into
`runs_index.csv` — so that alone decides the split. Separately, a handful
of `pollu`/`vanderpol`/`hires`/`orego` shooting seeds never finished on the
cluster at all within the wall-clock limit; those show up below as
`n_total < 10`, distinct from "finished but blew up.")

In [3]:
BLOWUP_THRESHOLD = 1.0  # E_trajectory_test of the Yhat=0 (predict-nothing) baseline

COMBOS = [
    ("StiffNet + Collocation", "stiff", "collocation"),
    ("GELU + Shooting", "GELU-scaled", "shooting"),
    ("GELU + Collocation", "GELU-scaled", "collocation"),
]

PROBLEM_DISPLAY = {
    "vanderpol": "Van der Pol",
    "pollu": "POLLU",
    "rober": "ROBER",
    "orego": "OREGO",
    "hires": "HIRES",
    "davis-skodje": "Davis--Skodje",
}


def reliability_table(db, combos=COMBOS, problems=PROBLEMS, threshold=BLOWUP_THRESHOLD,
                       n_attempted=N_ATTEMPTED):
    """For each (method, problem) cell report:
      - n_total: # of runs that actually finished on the cluster (<= n_attempted)
      - n_success: # of finished runs with 0 <= E_trajectory_test < threshold
      - median_error: median E_trajectory_test among the successful runs
      - avg_training_time: mean `training_time` (derivmatch + the main
        stage) over ALL finished runs, successful or not -- how long
        training took doesn't depend on whether the fit turned out usable.

    n_success is always reported out of `n_attempted` (10 seeds), not out of
    n_total: a seed that never finished on the cluster (wall-clock timeout)
    is not a successful run either, and folding that in keeps cluster
    failures from silently inflating the reliability numbers.
    """
    rows = []
    for method_label, model, training in combos:
        for problem in problems:
            grp = db[(db["problem"] == problem) & (db["model"] == model) & (db["training"] == training)]
            ok = grp[(grp["E_trajectory_test"] >= 0) & (grp["E_trajectory_test"] < threshold)]
            rows.append({
                "method": method_label,
                "problem": problem,
                "n_attempted": n_attempted,
                "n_total": len(grp),
                "n_missing": n_attempted - len(grp),
                "n_success": len(ok),
                "median_error": ok["E_trajectory_test"].median() if len(ok) else math.nan,
                "avg_training_time": grp["training_time"].mean() if len(grp) else math.nan,
            })
    return pd.DataFrame(rows)


table = reliability_table(DB)
table

,method,problem,n_attempted,n_total,n_missing,n_success,median_error,avg_training_time
0,StiffNet + Collocation,vanderpol,10,10,0,3,0.415318,63.556340
1,StiffNet + Collocation,pollu,10,10,0,9,0.005167,5661.213690
2,StiffNet + Collocation,rober,10,9,1,3,0.667507,50.044971
3,StiffNet + Collocation,orego,10,10,0,2,0.582513,98.070480
4,StiffNet + Collocation,hires,10,10,0,0,NaN,182.018473
5,StiffNet + Collocation,davis-skodje,10,10,0,0,NaN,168.030536
6,GELU + Shooting,vanderpol,10,9,1,1,0.543668,1521.301576
7,GELU + Shooting,pollu,10,0,10,0,NaN,NaN
8,GELU + Shooting,rober,10,10,0,0,NaN,398.223340
9,GELU + Shooting,orego,10,9,1,5,0.932860,5544.041130


`n_missing` above (seeds that never finished on the cluster, e.g. the
`pollu`+shooting wall-clock timeouts) is worth a glance before trusting the
LaTeX table's `# Successful Runs` column at face value -- a large
`n_missing` means that combo's reliability is (at least partly) an
infrastructure/budget question, not purely a numerics-blew-up one.

In [4]:
def format_time(seconds):
    if seconds is None or (isinstance(seconds, float) and math.isnan(seconds)):
        return "--"
    if seconds < 120:
        return f"{seconds:.0f}s"
    if seconds < 3600:
        return f"{seconds / 60:.1f} min"
    return f"{seconds / 3600:.2f} h"


def format_error(value):
    return "--" if (value is None or (isinstance(value, float) and math.isnan(value))) else f"{value:.3e}"


display_table = table.copy()
display_table["Median Error"] = display_table["median_error"].map(format_error)
display_table["Avg. Training Time"] = display_table["avg_training_time"].map(format_time)
display_table["# Successful Runs"] = (
    display_table["n_success"].astype(str) + "/" + display_table["n_attempted"].astype(str)
)
display_table["Problem"] = display_table["problem"].map(PROBLEM_DISPLAY)
display_table = display_table.set_index(["method", "Problem"])[
    ["# Successful Runs", "Median Error", "Avg. Training Time", "n_missing"]
]
display_table

# Successful Runs Median Error  \
method                 Problem                                        
StiffNet + Collocation Van der Pol                3/10    4.153e-01   
                       POLLU                      9/10    5.167e-03   
                       ROBER                      3/10    6.675e-01   
                       OREGO                      2/10    5.825e-01   
                       HIRES                      0/10           --   
                       Davis--Skodje              0/10           --   
GELU + Shooting        Van der Pol                1/10    5.437e-01   
                       POLLU                      0/10           --   
                       ROBER                      0/10           --   
                       OREGO                      5/10    9.329e-01   
                       HIRES                      0/10           --   
                       Davis--Skodje              0/10           --   
GELU + Collocation     Van der Pol                2/10    5.813e-01   
                       POLLU                      0/10           --   
                       ROBER                      0/10           --   
                       OREGO                      0/10           --   
                       HIRES                      0/10           --   
                       Davis--Skodje              0/10           --   

                                     Avg. Training Time  n_missing  
method                 Problem                                      
StiffNet + Collocation Van der Pol                  64s          0  
                       POLLU                     1.57 h          0  
                       ROBER                        50s          1  
                       OREGO                        98s          0  
                       HIRES                    3.0 min          0  
                       Davis--Skodje            2.8 min          0  
GELU + Shooting        Van der Pol             25.4 min          1  
                       POLLU                         --         10  
                       ROBER                    6.6 min          0  
                       OREGO                     1.54 h          1  
                       HIRES                     2.16 h          0  
                       Davis--Skodje           10.5 min          0  
GELU + Collocation     Van der Pol                  72s          0  
                       POLLU                     1.77 h          0  
                       ROBER                        44s          1  
                       OREGO                        98s          2  
                       HIRES                    3.5 min          0  
                       Davis--Skodje            5.0 min          0

In [5]:
def render_reliability_latex(table, combos=COMBOS, problems=PROBLEMS,
                              problem_display=PROBLEM_DISPLAY,
                              label="tab:reliability-ablation",
                              caption="Reliability of training with different combinations of architectures and training methods."):
    """Render `table` (the output of reliability_table) as the LaTeX
    table* the user specified: one \\multirow block per method, one row per
    problem, columns for successful-run count / median error / avg time.
    """
    lines = [
        r"\begin{table*}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        "",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{ll|ccc}",
        r"\toprule",
        r"\textbf{Training Method}",
        r"& \textbf{Problem}",
        r"& \textbf{\# Successful Runs}",
        r"& \textbf{Median Error}",
        r"& \textbf{Avg. Training Time} \\",
        r"\midrule",
        "",
    ]

    for i, (method_label, model, training) in enumerate(combos):
        lines.append(rf"\multirow{{{len(problems)}}}{{*}}{{{method_label}}}")
        for problem in problems:
            row = table[(table["method"] == method_label) & (table["problem"] == problem)].iloc[0]
            n_success = f'{int(row["n_success"])}/{int(row["n_attempted"])}'
            med = format_error(row["median_error"])
            t = format_time(row["avg_training_time"])
            label_text = problem_display.get(problem, problem)
            lines.append(rf"& {label_text:14s} & {n_success} & {med} & {t} \\")
        if i < len(combos) - 1:
            lines.append(r"\midrule")
        lines.append("")

    lines += [
        r"\bottomrule",
        r"\end{tabular}%",
        r"}",
        r"\end{table*}",
    ]
    return "\n".join(lines)


print(render_reliability_latex(table))

\begin{table*}[t]
\centering
\caption{Reliability of training with different combinations of architectures and training methods.}
\label{tab:reliability-ablation}

\resizebox{\textwidth}{!}{%
\begin{tabular}{ll|ccc}
\toprule
\textbf{Training Method}
& \textbf{Problem}
& \textbf{\# Successful Runs}
& \textbf{Median Error}
& \textbf{Avg. Training Time} \\
\midrule

\multirow{6}{*}{StiffNet + Collocation}
& Van der Pol    & 3/10 & 4.153e-01 & 64s \\
& POLLU          & 9/10 & 5.167e-03 & 1.57 h \\
& ROBER          & 3/10 & 6.675e-01 & 50s \\
& OREGO          & 2/10 & 5.825e-01 & 98s \\
& HIRES          & 0/10 & -- & 3.0 min \\
& Davis--Skodje  & 0/10 & -- & 2.8 min \\
\midrule

\multirow{6}{*}{GELU + Shooting}
& Van der Pol    & 1/10 & 5.437e-01 & 25.4 min \\
& POLLU          & 0/10 & -- & -- \\
& ROBER          & 0/10 & -- & 6.6 min \\
& OREGO          & 5/10 & 9.329e-01 & 1.54 h \\
& HIRES          & 0/10 & -- & 2.16 h \\
& Davis--Skodje  & 0/10 & -- & 10.5 min \\
\midrule

\multirow{6}{